In [1]:
import tensorflow as tf
import keras
from keras.layers import LSTM, Dense, Dropout
from keras.models import Sequential
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import pickle

## _. 使用停止中

#### 元データの取り込み

In [ ]:
data = pd.read_csv('weekly_All_v1.csv')

#### 使用するデータだけ抜き出す

In [ ]:
data = data[data['week_id'] >= 40]

In [ ]:
# 正解ラベルを準備する
data['next_close'] = data.groupby('nikkei')['close'].shift(-1)
data['next_return'] = data.groupby('nikkei')['return'].shift(-1)

In [ ]:
# 正解ラベルが存在しないため，最後尾の週は削除する
data = data.groupby('nikkei').apply(lambda x: x[:-1]).reset_index(drop=True)

In [ ]:
data.to_csv('weekly_All_v2.csv', index=False)

#### その前に，データを切り分ける

In [ ]:
data = data.sort_values(['week_id', 'nikkei'])

In [ ]:
data = data.rename(columns={
    '１株当たり配当金': 'DPS',
    'その他流動資産／その他の金融資産':'その他流動資産',
    'その他流動負債／その他の金融負債':'その他流動負債',
    '法人税等調整額／繰延税金費用':'法人税等調整額',
    'その他無形固定資産／その他無形資産':'その他無形固定資産',
    'その他の小計欄より上の営業キャッシュフロー':'その他の小計欄より上の営業CF',
    '投資有価証券の取得による支出（▲）': '投資有価証券の取得による支出',
    'その他の流動資産の減少額（▲増）': 'その他の流動資産の減少額',
})

In [ ]:
data.reset_index(drop=True, inplace=True)
data

In [ ]:
data[data['nikkei'] == 'N0056747']

In [ ]:
data['nikkei'].nunique()

In [ ]:
data.columns

In [ ]:
tech_order =[
    '上場フラグ',
    'nikkei',
    'week_id',
    'return',
    '10mom',
    'rsi14',
    'signal',
    'macd',
    'hist',
    'slowk',
    'slowd',
    '5ma',
    '15ma',
    '40ma',
    'bb1u',
    'bb1l',
    'bb2u',
    'bb2l',
    'beta',
    'volatility',
]

In [ ]:
funda_order = [
    '上場フラグ',
    'nikkei',
    'week_id',

    # 財務指標
    '固定比率',
    '流動比率',
    '売上高総利益率',
    '売上高営業利益率',
    '売上高経常利益率',
    '株主資本回転率',
    '総資産回転率',
    'ギアリング比率',
    '自己資本比率',
    'フリーキャッシュフロー',
    '有利子負債',
    'ROE',
    'ROA',
    'PER',
    'PBR',
    'PCFR',
    'EPS',
    'BPS',
    'DPS',
    'CFPS',
    'EV',
    'EBITDA',
]

In [ ]:
fiscal_order = [
    '上場フラグ',
    'nikkei',
    'week_id',

    # BS
    'その他流動資産',
    'その他流動負債',
    'その他無形固定資産',

    # PL
    '法人税等調整額',
    'その他営業外費用',
    'その他営業外収益',
    '営業利益',
    '営業外費用',
    '営業費用',
    '非支配株主に帰属する当期純利益',
    'その他特別利益',

    # CF
    '有形固定資産の売却による収入',
    '無形固定資産の売却による収入',
    '固定資産の売却による収入',
    '投資有価証券の取得による支出',
    '投資その他の資産の売却による収入',
    'その他の流動資産の減少額',
    'その他の小計欄より上の営業CF',
]

In [ ]:
all_data = [
    '上場フラグ',
    'nikkei',
    'week_id',

    # tech
    'return',
    '10mom',
    'rsi14',
    'signal',
    'macd',
    'hist',
    'slowk',
    'slowd',
    '5ma',
    '15ma',
    '40ma',
    'bb1u',
    'bb1l',
    'bb2u',
    'bb2l',
    'beta',
    'volatility',

    # fundamental
    '固定比率',
    '流動比率',
    '売上高総利益率',
    '売上高営業利益率',
    '売上高経常利益率',
    '株主資本回転率',
    '総資産回転率',
    'ギアリング比率',
    '自己資本比率',
    'フリーキャッシュフロー',
    '有利子負債',
    'ROE',
    'ROA',
    'PER',
    'PBR',
    'PCFR',
    'EPS',
    'BPS',
    'DPS',
    'CFPS',
    'EV',
    'EBITDA',

    # fiscal
    # - BS
    'その他流動資産',
    'その他流動負債',
    'その他無形固定資産',

    # - PL
    '法人税等調整額',
    'その他営業外費用',
    'その他営業外収益',
    '営業利益',
    '営業外費用',
    '営業費用',
    '非支配株主に帰属する当期純利益',
    'その他特別利益',

    # - CF
    '有形固定資産の売却による収入',
    '無形固定資産の売却による収入',
    '固定資産の売却による収入',
    '投資有価証券の取得による支出',
    '投資その他の資産の売却による収入',
    'その他の流動資産の減少額',
    'その他の小計欄より上の営業CF',
]

In [ ]:
tech_df     = data[tech_order   + ['next_return', 'next_close']]
funda_df    = data[funda_order  + ['next_return', 'next_close']]
fiscal_df   = data[fiscal_order + ['next_return', 'next_close']]
all_df      = data[all_data     + ['next_return', 'next_close']]

In [ ]:
# fiscal_corr = fiscal_df.corr().unstack().abs().reset_index()
# fiscal_corr.columns = ['feature_1', 'feature_2', 'correlation']
# fiscal_corr = fiscal_corr[fiscal_corr['correlation'].abs() < 1.0]
# fiscal_corr = fiscal_corr.sort_values(by='correlation', ascending=False)
# fiscal_corr = fiscal_corr.reset_index(drop=True)
# fiscal_corr = fiscal_corr[fiscal_corr.index % 2 == 0].reset_index(drop=True)
# fiscal_corr.head(40)

#### データを使用するフォーマットに成形

In [ ]:
all_wed_csv = pd.read_csv('weekly_All_wed_v2.csv')

In [ ]:
all_df_with_yearID = pd.merge(
    all_df,
    all_wed_csv[['week_id', 'separate_flag']],
    on='week_id',
    how='left'
)

In [ ]:
all_df_with_yearID.groupby('separate_flag').apply(lambda group: group.to_csv(f'each_year_data/all_df_{group.name}.csv', index=False))

In [ ]:
csv_list = [pd.read_csv(fp) for fp in Path('each_year_data').rglob('.') if fp.suffix == '.csv']

input_csv_set = [
    pd.concat([csv1, csv2, csv3, csv4], axis=0).reset_index(drop=True)
    for csv1, csv2, csv3, csv4, in zip(
        csv_list[1:-3],
        csv_list[2:-2],
        csv_list[3:-1],
        csv_list[4:],
    )
]

In [ ]:
input_csv_set = [csv.sort_values(['nikkei', 'week_id']).reset_index(drop=True) for csv in input_csv_set]

In [ ]:
len(input_csv_set)

In [ ]:
input_csv_set[0]

In [ ]:
for i, input_csv in enumerate(input_csv_set):
    print('\r', f'Processing set {i+1}-{i+4}', end='   ')
    # input_csv = input_csv.drop(columns=['nikkei', 'week_id'], errors='ignore')
    input_csv = input_csv.fillna(0, inplace=False)
    input_csv.to_csv(f'every_3year_data/input_csv_set_{i+2}-{i+5}.csv', index=False)

## 1. モデルを記述（必ず実行）

#### モデルの具体的な設定

＠ notebook\tmp\create_simple_LSTM.ipynb

ここに元のモデルが記述されている

In [2]:
lstm_cfg = {
    'neurons_in_layer' : 30,
    'learning_rate' : 0.001,
    'decay' : 0,
    'hidden_layers' : 1,
    'optimizer' : keras.optimizers.RMSprop(learning_rate=0.001, decay=0),
    'early_stopping' : 10,
    'loss_function' : 'binary_crossentropy',
    'epochs' : 100,
    'dropout_rate' : 0.5,
    'batch_size' : 32,
}

c:\Users\Ito Yuki\FINAL_PAPER\research_using_uv\.venv\Lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


In [8]:
def build_and_train_LSTM(X_train: np.ndarray, y_train: np.ndarray):
    model = Sequential()
    # 1層目
    model.add(LSTM(units=lstm_cfg['neurons_in_layer'], return_sequences=True, 
                   input_shape=(X_train.shape[1], X_train.shape[2])))
    model.add(Dropout(lstm_cfg['dropout_rate']))
    # 2層目
    model.add(LSTM(units=lstm_cfg['neurons_in_layer'], return_sequences=False))
    # 出力層
    model.add(Dense(units=1))

    model.compile(optimizer=lstm_cfg['optimizer'], loss=lstm_cfg['loss_function'])
    model.fit(X_train, y_train,
              epochs=lstm_cfg['epochs'],
              batch_size=lstm_cfg['batch_size'],
              validation_split=0.2,
              callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=lstm_cfg['early_stopping'])])
    return model

## 2. データのインポート

- weekly_all_v2.csv

In [ ]:
input_csv = pd.read_csv('weekly_all_v2.csv')

#### 検証データと訓練データの作成

In [4]:
train_df = pd.read_csv('input/train_df.csv')
test_df = pd.read_csv('input/test_df.csv')

In [ ]:
train_df

In [ ]:
len(train_df.drop(columns=['next_return', 'next_close', 'nikkei', 'week_id', 'separate_flag'], errors='ignore').columns.tolist())

#### モデルの実行

In [5]:
def prepare_LSTM_data(train_df: pd.DataFrame):
    # パラメータ
    timesteps = 30

    train_df = train_df.sort_values(['nikkei', 'week_id']).reset_index(drop=True)

    data = train_df.drop(
        columns=['next_return', 'nikkei', 'week_id', 'separate_flag']
        ).values

    data = data.reshape(4377, 951, 59)  # (銘柄数, 日数, 特徴量数)

    num_stocks, num_days, num_features = data.shape

    X_list = []
    y_list = []

    for stock_idx in range(num_stocks):
        for i in range(num_days - timesteps):
            X_list.append(data[stock_idx, i:i+timesteps, :])  # 過去 timesteps 日
            y_list.append(data[stock_idx, i+timesteps, -1])    # 予測する値（例：終値）

    # リストを NumPy 配列に変換
    X_train = np.array(X_list, dtype=np.float32)
    y_train = np.array(y_list, dtype=np.float32).reshape(-1, 1)

    print("X_train.shape:", X_train.shape)
    print("y_train.shape:", y_train.shape)

    return X_train, y_train

In [6]:
def LSTM_pipeline(nikkei_train_group: pd.DataFrame, nikkei_test_group: pd.DataFrame, y_label:str):
    X, y = prepare_LSTM_data(nikkei_train_group)
    model = build_and_train_LSTM(X, y)
    with open(f'../../model/dev_LSTM/model.pkl', 'wb') as f:
        pickle.dump(model, f)

    test_X, test_y = prepare_LSTM_data(nikkei_test_group)
    pred = model.predict(test_X)

    plt.figure(figsize=(12,6))
    plt.plot(test_y, color='blue', label='Actual')
    plt.plot(pred, color='red', label='Predicted')
    plt.legend()
    plt.title(f'LSTM Prediction vs Actual')
    plt.xlabel('Time')
    plt.ylabel(y_label)
    plt.savefig(f'../../images/dev_LSTM/prediction.png')
    plt.show()

    ret = pd.DataFrame({
        'actual': test_y.flatten(),
        'predicted': pred.flatten()
    })

    return ret

In [9]:
LSTM_pipeline(train_df, test_df, y_label='next_close')

X_train.shape: (4031217, 30, 59)
y_train.shape: (4031217, 1)


c:\Users\Ito Yuki\FINAL_PAPER\research_using_uv\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


KeyboardInterrupt: 